In [ ]:
#instalação das bibliotecas necessárias
!pip install ultralytics roboflow -q

In [ ]:
#importações dos recursos
from roboflow import Roboflow
from ultralytics import YOLO
import os
import random
import shutil

In [ ]:
#download dos dados (a partir do google drive) 
dataset_path = "/content/drive/MyDrive/dataset"
print(os.listdir(dataset_path))

In [ ]:
#caminhos
base = "/content/drive/MyDrive/dataset"

images_dir = os.path.join(base, "images")
labels_dir = os.path.join(base, "labels")

train_img = os.path.join(base, "images/train")
val_img = os.path.join(base, "images/val")

train_lbl = os.path.join(base, "labels/train")
val_lbl = os.path.join(base, "labels/val")

In [ ]:
#criação das pastas
for p in [train_img, val_img, train_lbl, val_lbl]:
    os.makedirs(p, exist_ok=True)

#listagem das imagens
images = [
    f for f in os.listdir(images_dir)
    if f.endswith((".jpg", ".jpeg", ".png"))
]

In [ ]:
#embaralhar imagens
random.shuffle(images)

#split (treino e validação)
split = int(len(images) * 0.8)

train = images[:split]
val = images[split:]

In [ ]:
#função para fazer as anotações
def copy_files(files, img_dest, lbl_dest):

    for img in files:

        label = os.path.splitext(img)[0] + ".txt"

        src_img = os.path.join(images_dir, img)
        src_lbl = os.path.join(labels_dir, label)

        dst_img = os.path.join(img_dest, img)
        dst_lbl = os.path.join(lbl_dest, label)

        shutil.copy(src_img, dst_img)

        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)

copy_files(train, train_img, train_lbl)
copy_files(val, val_img, val_lbl)

print("Dataset organizado!")

In [ ]:
#criação do data.yaml
yaml_content = """
path: /content/drive/MyDrive/dataset

train: images/train
val: images/val

names:
  0: fissura
"""

with open("/content/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml criado!")

In [ ]:
#instância do modelo
model = YOLO("yolov8n-seg.pt")

In [ ]:
#treino do modelo
model.train(
    data="/content/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8
)

In [ ]:
#teste do modelo
results = model("img3-f.jpg", conf=0.5, iou=0.5)

count = 0
for r in results:
    count = len(r.boxes)

print("Quantidade de fissuras:", count)